In [4]:
import pandas as pd
import numpy as np
import os

# Thêm 'data_raw/' vào trước tên file
current_dir = os.getcwd()

# 2. Nhảy ra ngoài thư mục cha (thư mục gốc của dự án) rồi mới vào 'data'
# '..' nghĩa là đi ngược lên một cấp thư mục
path = os.path.join(current_dir, '..', 'data_raw', '2022.csv')
df = pd.read_csv(path)

print("--- Đã đọc xong file 2022.csv ---")
print(f"Số dòng ban đầu: {len(df)}")

# 3. CHIA TÁCH CỘT NGÀY GIỜ (Feature Engineering)
# Chuyển sang định dạng datetime
df['Local Time'] = pd.to_datetime(df['Local Time'])

# Tách riêng biệt từng cột
df['Date'] = df['Local Time'].dt.date          # YYYY-MM-DD
df['Year'] = df['Local Time'].dt.year          # Năm
df['Month'] = df['Local Time'].dt.month        # Tháng
df['Hour'] = df['Local Time'].dt.hour          # Giờ (0-23)
df['DayOfWeek'] = df['Local Time'].dt.dayofweek # Thứ (0=Thứ 2, 6=CN)
df['IsWeekend'] = df['DayOfWeek'].apply(lambda x: 1 if x >= 5 else 0) # Biến cuối tuần

# 4. XỬ LÝ LỖI DỮ LIỆU (Cleaning)
# Nội suy các giá trị Missing (xử lý khoảng trống ở 2022)
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].interpolate(method='linear')

# Xử lý Outliers cho CO (Chặn ngưỡng 2000 như đã bàn)
if 'CO' in df.columns:
    df['CO'] = df['CO'].clip(upper=2000)

# Xử lý đồng bộ tên cột (Country Code và Timezone có file viết hoa, file viết thường)
df.columns = [c.lower().replace(' ', '_') for c in df.columns]

# 5. SẮP XẾP LẠI THỨ TỰ CỘT CHO ĐẸP (Đưa thời gian lên đầu)
time_cols = ['local_time', 'date', 'year', 'month', 'hour', 'dayofweek', 'isweekend']
other_cols = [c for c in df.columns if c not in time_cols]
df = df[time_cols + other_cols]


print("\n--- KẾT QUẢ ---")
print(f"Số lượng giá trị thiếu còn lại: {df.isnull().sum().sum()}")
print("Các cột hiện có trong file sạch:")
print(df.columns.tolist())
df.head()

--- Đã đọc xong file 2022.csv ---
Số dòng ban đầu: 8472

--- KẾT QUẢ ---
Số lượng giá trị thiếu còn lại: 0
Các cột hiện có trong file sạch:
['local_time', 'date', 'year', 'month', 'hour', 'dayofweek', 'isweekend', 'utc_time', 'city', 'country_code', 'timezone', 'aqi', 'co', 'no2', 'o3', 'pm10', 'pm25', 'so2', 'clouds', 'precipitation', 'pressure', 'relative_humidity', 'temperature', 'uv_index', 'wind_speed']


,local_time,date,year,month,hour,dayofweek,isweekend,utc_time,city,country_code,...,pm10,pm25,so2,clouds,precipitation,pressure,relative_humidity,temperature,uv_index,wind_speed
0,2022-01-13 00:00:00,2022-01-13,2022,1,0,3,0,2022-01-12T17:00:00,Hanoi,VN,...,109.3,77.67,54.7,100,0.0,1019,93,17.0,0.0,1.22
1,2022-01-13 01:00:00,2022-01-13,2022,1,1,3,0,2022-01-12T18:00:00,Hanoi,VN,...,111.0,79.00,59.0,100,0.0,1019,97,16.8,0.0,1.33
2,2022-01-13 02:00:00,2022-01-13,2022,1,2,3,0,2022-01-12T19:00:00,Hanoi,VN,...,107.3,76.33,58.7,91,0.0,1019,97,16.4,0.0,1.44
3,2022-01-13 03:00:00,2022-01-13,2022,1,3,3,0,2022-01-12T20:00:00,Hanoi,VN,...,103.7,73.67,58.3,83,0.0,1019,97,16.1,0.0,3.10
4,2022-01-13 04:00:00,2022-01-13,2022,1,4,3,0,2022-01-12T21:00:00,Hanoi,VN,...,100.0,71.00,58.0,75,0.0,1018,97,15.7,0.0,1.66
